In [ ]:
import torch
import torch.nn as nn
from MSG_GAN.vectorised_encoder import VectorisedEncoder
from MSG_GAN.segmentation_generator import SegmentationGenerator
enc = VectorisedEncoder(z_dim=256,
                          channels=[512, 512, 256, 128, 64, 32, 16],
                          number_of_vectorizers=3,
                          vector_dim=256,
                          degrees_of_freedom=8,
                          vectorizer_linear_dim=[256,256],
                          img_channels=3,
                          block_depth=2,
                          residual=True,
                          use_norm=True)


import torch.nn as nn
import torch.nn.functional as F

from MSG_GAN.generator import Generator    



class SegmentationGenerator(nn.Module):
    def __init__(self, 
                z_dim,
                channels,
                number_of_vectorizers,
                img_channels=3,
                block_depth=2,
                residual=True,
                use_norm=True,
                final_activation=None):
        super().__init__()
        self.channels = channels.copy()
        self.number_of_vectorizers = number_of_vectorizers
        self.generator = Generator(z_dim=z_dim,
                                   channels=self.channels,
                                   img_channels=img_channels,
                                   block_depth=block_depth,
                                   residual=residual,
                                   use_norm=use_norm)
        self.final_activation = final_activation
    
    def forward(self, x):
        number_of_vectorizers = x.shape[0]
        assert number_of_vectorizers == self.number_of_vectorizers
        b_size = x.shape[1]
        print(number_of_vectorizers,b_size)


        x = x.view(b_size*number_of_vectorizers,*x.shape[2:])
        print(x.shape)
        x = self.generator(x)
        print(x.shape)
        x = x.view(self.number_of_vectorizers,b_size,*x.shape[2:])
        return x

seg_gen = SegmentationGenerator(z_dim=256,
                                channels=[512, 512, 256, 128, 64, 32, 16],
                                number_of_vectorizers=3,        
                                img_channels=3,
                                block_depth=2,
                                residual=True,
                                use_norm=True,
                                final_activation=nn.Softmax(dim=1))
x = torch.randn(4,3,256,256)
y = enc(x)

y = seg_gen(y)


3 4
torch.Size([12, 256, 1, 1])


AttributeError: 'list' object has no attribute 'shape'

In [9]:
y.shape[1:]

torch.Size([4, 256, 1, 1])